# 09 · ExcelFormer Baseline (Lending Club)

ExcelFormer — a transformer architecture designed specifically for tabular data — applied to the Lending Club default-risk task. The model combines:

- **Semi-permeable attention** (DIAM / AIUM gates)
- **Feat-mix / hidden-mix** augmentation
- **CatBoost-style** target-statistics encoding for categoricals

Used as a non-FT-Transformer transformer baseline. The original notebook did not include its own preprocessing — the standard LC preprocessing block is added here so the notebook runs standalone.

## 1. Setup

In [ ]:
# %pip install rtdl_revisiting_models -q
%pip install delu -q
%pip install optuna -q

In [ ]:
import math
import random
import warnings
import json
import os
import itertools
import typing
from collections import OrderedDict
from pathlib import Path
from typing import Any, Dict, Iterable, List, Literal, Optional, Tuple, cast

import numpy as np
import pandas as pd
import scipy.special

import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim
from torch import Tensor
from torch.nn.parameter import Parameter
from torch.utils.data import Dataset, DataLoader, TensorDataset

import sklearn.datasets
import sklearn.metrics
import sklearn.model_selection
import sklearn.preprocessing
import sklearn.tree as sklearn_tree
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler, LabelEncoder, QuantileTransformer
from sklearn.metrics import roc_auc_score
from sklearn.isotonic import IsotonicRegression

import optuna
import delu
from tqdm.std import tqdm
from tqdm import tqdm as _tqdm  # alias for cells that use it directly

warnings.filterwarnings("ignore")
pd.options.display.max_rows = 200
pd.options.display.max_columns = 200

RANDOM_SEED = 42


def seed_everything(seed: int = RANDOM_SEED) -> None:
    random.seed(seed)
    np.random.seed(seed)
    torch.manual_seed(seed)
    if torch.cuda.is_available():
        torch.cuda.manual_seed_all(seed)


seed_everything()
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Device: {device}")

In [ ]:
# --- Optional: Google Colab Drive mount ---
# Uncomment the three lines below if you're running on Colab and want to read
# data from / write artifacts to your Drive. Skip on local, server, or Kaggle
# runs. The next cell auto-routes through DRIVE_ROOT when defined.
#
# from google.colab import drive
# drive.mount("/content/drive")
# DRIVE_ROOT = "/content/drive/MyDrive/ft-transformer-credit-risk-study"

# When running locally, repo root is one directory up (notebook is in
# notebooks/<dataset>/). On Colab with the cell above uncommented, DRIVE_ROOT
# takes precedence.
_BASE = globals().get("DRIVE_ROOT", "..")
DATA_PATH      = f"{_BASE}/data/processed_data_densest.parquet.gzip"
ARTIFACTS_DIR  = Path(f"{_BASE}/artifacts/lending_club")
RESULTS_DIR    = ARTIFACTS_DIR / "results"
MODELS_DIR     = ARTIFACTS_DIR / "models"
FIGURES_DIR    = ARTIFACTS_DIR / "figures"
for _d in (ARTIFACTS_DIR, RESULTS_DIR, MODELS_DIR, FIGURES_DIR):
    _d.mkdir(parents=True, exist_ok=True)

print(f"DATA_PATH      = {DATA_PATH}")
print(f"ARTIFACTS_DIR  = {ARTIFACTS_DIR}")

In [ ]:
# ──────────────────────────────────────────────────────────────────────────────
# DEV_MODE — smoke-test switch.
#
# When False (the default), every constant resolves to the exact value used in
# the original experiment. The DEV_MODE branches below are dead code in that
# path, so the run is behaviourally identical to the source notebook.
#
# When True, the notebook runs a tiny fast version: a subsample of the data,
# a couple of epochs, a single Optuna trial. Use this on Colab to validate the
# full pipeline end-to-end before launching a real run.
# ──────────────────────────────────────────────────────────────────────────────
DEV_MODE = False
if DEV_MODE:
    print("DEV_MODE is ON — smoke-test run with reduced data/epochs/trials.")

## 2. Data Loading

In [ ]:
# Load preprocessed Lending Club dataset.
df4 = pd.read_parquet(DATA_PATH)

train_val_df, test_df = train_test_split(
    df4, test_size=0.2,
    stratify=df4["target_binary"], random_state=RANDOM_SEED,
)
train_df, val_df = train_test_split(
    train_val_df, test_size=0.2,
    stratify=train_val_df["target_binary"], random_state=RANDOM_SEED,
)
print("Train_val / test sizes:", len(train_val_df), len(test_df))
print("Train / val sizes:", len(train_df), len(val_df))

cat_columns = [
    "term", "emp_length", "home_ownership", "verification_status",
    "purpose", "zip_code", "addr_state", "application_type",
    "initial_list_status", "disbursement_method",
]
num_cols = [
    c for c in list(df4.columns)
    if c not in cat_columns + ["id", "emp_title", "target_binary"]
]
print(f"# numerical features: {len(num_cols)}  |  # categorical: {len(cat_columns)}")

## 3. Preprocessing

In [ ]:
# Numeric imputation + scaling (median + StandardScaler).
num_impute = {}
scaler = StandardScaler()

num_df = train_df[num_cols].copy()
for c in num_df.columns:
    if num_df[c].dtype == "object":
        num_df[c] = pd.to_numeric(num_df[c], errors="coerce")
    med = num_df[c].median()
    num_df[c] = num_df[c].fillna(med)
    num_impute[c] = med
num_df[num_df.columns] = scaler.fit_transform(num_df[num_df.columns])

# Label encoding with a permanent UNKNOWN bucket for unseen validation/test categories.
cat_encoders: Dict[str, LabelEncoder] = {}
cat_cardinalities = []
cat_df = train_df[cat_columns].copy()
for c in cat_columns:
    le = LabelEncoder()
    series = cat_df[c].fillna("MISSING").astype(str)
    le.fit(series)
    new_classes = list(le.classes_)
    if "MISSING" not in new_classes:
        new_classes.append("MISSING")
    new_classes.append("UNKNOWN")
    le.classes_ = np.array(new_classes)
    cat_df[c + "_le"] = le.transform(series)
    cat_encoders[c] = le
    cat_cardinalities.append(len(le.classes_))

train_df_proc = pd.concat(
    [
        num_df.reset_index(drop=True),
        cat_df[[c + "_le" for c in cat_columns]].reset_index(drop=True),
        train_df["target_binary"].reset_index(drop=True),
    ],
    axis=1,
)
print("Encoded train")

# Apply the same imputation, scaling and label maps to val and test.
def _apply_transforms(part_df):
    num_part = part_df[num_cols].copy()
    for c in num_part.columns:
        if num_part[c].dtype == "object":
            num_part[c] = pd.to_numeric(num_part[c], errors="coerce")
    for c in num_part.columns:
        num_part[c] = num_part[c].fillna(num_impute[c])
    num_part = pd.DataFrame(
        scaler.transform(num_part), columns=num_part.columns, index=num_part.index
    )

    cat_part = part_df[cat_columns].copy()
    for c in cat_columns:
        le = cat_encoders[c]
        mapping = {cls: i for i, cls in enumerate(le.classes_)}
        unknown_id = mapping["UNKNOWN"]
        series = cat_part[c].fillna("MISSING").astype(str)
        cat_part[c + "_le"] = series.map(mapping).fillna(unknown_id).astype(int)

    return pd.concat(
        [
            num_part.reset_index(drop=True),
            cat_part[[c + "_le" for c in cat_columns]].reset_index(drop=True),
            part_df["target_binary"].reset_index(drop=True),
        ],
        axis=1,
    )


val_df_proc = _apply_transforms(val_df)
test_df_proc = _apply_transforms(test_df)
print("Encoded val and test")

In [ ]:
# Materialize everything as torch tensors on the target device.
cat_le_cols = [c + "_le" for c in cat_columns]
data_numpy = {"train": {}, "val": {}, "test": {}}
for part_name, part_df in [
    ("train", train_df_proc),
    ("val",   val_df_proc),
    ("test",  test_df_proc),
]:
    data_numpy[part_name]["X_cont"] = part_df[num_cols]
    data_numpy[part_name]["x_cat"]  = part_df[cat_le_cols]
    data_numpy[part_name]["y"]      = part_df["target_binary"]

data = {}
for part in data_numpy:
    data[part] = {
        "X_cont": torch.tensor(data_numpy[part]["X_cont"].to_numpy(), dtype=torch.float32, device=device),
        "x_cat":  torch.tensor(data_numpy[part]["x_cat"].to_numpy(),  dtype=torch.long,    device=device),
        "y":      torch.tensor(data_numpy[part]["y"].to_numpy(),      dtype=torch.float32, device=device),
    }

d_numerical = len(num_cols)
n_train = data["train"]["y"].shape[0]
print(f"d_numerical = {d_numerical}")
print(f"cat_cardinalities = {cat_cardinalities}")
print(f"# train rows = {n_train}")

In [ ]:
# ExcelFormer expects this naming convention used by the original code.
cat_cardinalities_final = cat_cardinalities

In [ ]:
# When DEV_MODE is on, subsample data to a few thousand rows so a full training
# pass completes in a couple of minutes. No-op when DEV_MODE is False.
if DEV_MODE:
    _n_dev = 5000
    for _part in data:
        _idx = torch.randperm(data[_part]["y"].shape[0], device=device)[:_n_dev]
        for _k in data[_part]:
            data[_part][_k] = data[_part][_k][_idx]
    print({_part: {k: v.shape for k, v in data[_part].items()} for _part in data})

## 4. Model Definition

ExcelFormer combines a `CatBoostEncoder` for categorical features, a linear `NumericalEmbedding` for continuous features, `SemiPermeableAttention` blocks, and either feat-mix or hidden-mix augmentation. The training-time mixup loss is provided by `mixup_bce`.

In [ ]:
class CatBoostEncoder(nn.Module):
    """
    Lightweight target-statistics encoder for categorical features
    (approximates the CatBoost-style encoder used in the paper).
    Learns a per-category embedding table, initialised to zero.
    """
    def __init__(self, cardinalities: List[int], out_dim: int):
        super().__init__()
        self.embeddings = nn.ModuleList([
            nn.Embedding(card + 1, out_dim, padding_idx=card)
            for card in cardinalities
        ])
        nn.init.zeros_(self.embeddings[0].weight) # start near zero
        for emb in self.embeddings:
            nn.init.normal_(emb.weight, std=0.01)

    def forward(self, x_cat: torch.Tensor) -> torch.Tensor:
        # x_cat: [B, C_cat] -> [B, C_cat, D]
        out = torch.stack([emb(x_cat[:, i]) for i, emb in enumerate(self.embeddings)], dim=1)
        return out


class NumericalEmbedding(nn.Module):
    """Linear embedding of each numerical feature into D-dimensional space."""
    def __init__(self, num_features: int, embed_dim: int):
        super().__init__()
        self.weight = nn.Parameter(torch.empty(num_features, embed_dim))
        self.bias = nn.Parameter(torch.zeros(num_features, embed_dim))
        nn.init.kaiming_uniform_(self.weight, a=math.sqrt(5))

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        # x: [B, F] -> [B, F, D]
        return x.unsqueeze(-1) * self.weight.unsqueeze(0) + self.bias.unsqueeze(0)


class SemiPermeableAttention(nn.Module):
    """
    Semi-Permeable Attention (SPA) — Eq. (2-4) in the paper.
    More informative features can influence less informative ones, but
    not vice versa (upper-triangular mask after MI-sort).
    Interaction-Attenuation Initialization sets Q/K/V weights near-zero.
    """
    def __init__(self, embed_dim: int, num_heads: int, dropout: float = 0.0):
        super().__init__()
        assert embed_dim % num_heads == 0
        self.embed_dim = embed_dim
        self.num_heads = num_heads
        self.head_dim = embed_dim // num_heads
        self.scale = self.head_dim ** -0.5
        self.dropout = nn.Dropout(dropout)

        self.q_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.k_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.v_proj = nn.Linear(embed_dim, embed_dim, bias=False)
        self.out_proj = nn.Linear(embed_dim, embed_dim, bias=False)

        # Interaction-Attenuation Initialization
        for proj in [self.q_proj, self.k_proj, self.v_proj]:
            nn.init.normal_(proj.weight, std=1e-4)
        nn.init.zeros_(self.out_proj.weight)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        """
        x : [B, C, D]
        mask: [C, C] upper-triangular causal mask (True = keep, False = block)
        """
        B, C, D = x.shape
        H, Hd = self.num_heads, self.head_dim

        q = self.q_proj(x).view(B, C, H, Hd).transpose(1, 2) # [B,H,C,Hd]
        k = self.k_proj(x).view(B, C, H, Hd).transpose(1, 2)
        v = self.v_proj(x).view(B, C, H, Hd).transpose(1, 2)

        attn = torch.matmul(q, k.transpose(-2, -1)) * self.scale # [B,H,C,C]

        if mask is not None:
            # mask: [C,C] bool – positions to MASK OUT set to -inf
            attn = attn.masked_fill(~mask.unsqueeze(0).unsqueeze(0), float("-inf"))

        attn = F.softmax(attn, dim=-1)
        attn = self.dropout(attn)

        out = torch.matmul(attn, v) # [B,H,C,Hd]
        out = out.transpose(1, 2).reshape(B, C, D)
        return self.out_proj(out)


class AttentiveFFN(nn.Module):
    """
    Attentive Feedforward (AIUM) — boosts fitting capability (Problem P3).
    Uses a gating mechanism: output = gate(x) ⊙ transform(x).
    """
    def __init__(self, embed_dim: int, expansion: int = 4, dropout: float = 0.0):
        super().__init__()
        hidden = embed_dim * expansion
        self.fc1 = nn.Linear(embed_dim, hidden)
        self.gate = nn.Linear(embed_dim, hidden)
        self.fc2 = nn.Linear(hidden, embed_dim)
        self.drop = nn.Dropout(dropout)

    def forward(self, x: torch.Tensor) -> torch.Tensor:
        g = torch.sigmoid(self.gate(x))
        h = F.gelu(self.fc1(x)) * g
        return self.fc2(self.drop(h))


class ExcelFormerBlock(nn.Module):
    """One ExcelFormer Transformer block: SPA + AIUM with pre-LN."""
    def __init__(self, embed_dim: int, num_heads: int,
                 diam_dropout: float = 0.0,
                 aium_dropout: float = 0.0,
                 residual_dropout: float = 0.0):
        super().__init__()
        self.ln1 = nn.LayerNorm(embed_dim)
        self.attn = SemiPermeableAttention(embed_dim, num_heads, diam_dropout)
        self.ln2 = nn.LayerNorm(embed_dim)
        self.ffn = AttentiveFFN(embed_dim, dropout=aium_dropout)
        self.drop = nn.Dropout(residual_dropout)

    def forward(self, x: torch.Tensor, mask: Optional[torch.Tensor] = None) -> torch.Tensor:
        x = x + self.drop(self.attn(self.ln1(x), mask))
        x = x + self.drop(self.ffn(self.ln2(x)))
        return x


class ExcelFormer(nn.Module):
    """
    Full ExcelFormer model.

    Architecture:
        1. Embed numerical and categorical features -> [B, C_total, D]
        2. MI-based feature sort (approximated by learnable sort during training)
        3. Stack of ExcelFormerBlock with SPA mask
        4. Attentive decoder -> logits
    """
    def __init__(
        self,
        num_numerical: int,
        cat_cardinalities: List[int],
        embed_dim: int = 64,
        num_layers: int = 3,
        num_heads: int = 8,
        out_channels: int = 1, # single logit → BCE, matching d_out=1 style
        diam_dropout: float = 0.1,
        aium_dropout: float = 0.1,
        residual_dropout: float = 0.0,
        mixup: Optional[str] = "feat_mix", # "feat_mix" | "hidden_mix" | None
        beta: float = 0.5,
    ):
        super().__init__()
        self.num_numerical = num_numerical
        self.mixup = mixup
        self.beta = beta
        total_cols = num_numerical + len(cat_cardinalities)
        self.total_cols = total_cols

        # ── encoders ─────────────────────────────────────────────────────────
        self.num_embed = NumericalEmbedding(num_numerical, embed_dim)
        self.cat_embed = CatBoostEncoder(cat_cardinalities, embed_dim) if cat_cardinalities else None

        # Learnable MI-sort permutation (fixed after training in the paper;
        # here we use a soft permutation via temperature-scaled scores)
        self.feature_scores = nn.Parameter(torch.randn(total_cols))

        # ── transformer blocks ────────────────────────────────────────────────
        self.blocks = nn.ModuleList([
            ExcelFormerBlock(embed_dim, num_heads, diam_dropout, aium_dropout, residual_dropout)
            for _ in range(num_layers)
        ])

        # ── decoder (attentive linear as in the paper) ────────────────────────
        self.ln_out = nn.LayerNorm(embed_dim)
        self.attn_w = nn.Linear(embed_dim, 1)
        self.head = nn.Linear(embed_dim, out_channels)

        self.register_buffer("spa_mask", self._build_spa_mask(total_cols))

    # ── helpers ───────────────────────────────────────────────────────────────
    @staticmethod
    def _build_spa_mask(n: int) -> torch.Tensor:
        """Upper-triangular mask: feature i can attend to feature j only if j <= i
        (after sorting from most→least informative)."""
        return torch.ones(n, n, dtype=torch.bool).tril()

    def _feat_mix(self, x: torch.Tensor, y: torch.Tensor):
        """Feat-Mix: mix raw feature embeddings between two samples."""
        if not self.training:
            return x, y, None
        lam = np.random.beta(self.beta, self.beta)
        B = x.size(0)
        idx = torch.randperm(B, device=x.device)
        x_mix = lam * x + (1 - lam) * x[idx]
        y_mix = (y, y[idx], lam)
        return x_mix, y_mix, idx

    def _hidden_mix(self, x: torch.Tensor, y: torch.Tensor):
        """Hidden-Mix: mix hidden representations mid-forward."""
        if not self.training:
            return x, y, None
        lam = np.random.beta(self.beta, self.beta)
        B = x.size(0)
        idx = torch.randperm(B, device=x.device)
        x_mix = lam * x + (1 - lam) * x[idx]
        return x_mix, (y, y[idx], lam), idx

    def forward(self, x_num: torch.Tensor, x_cat: Optional[torch.Tensor] = None,
                y: Optional[torch.Tensor] = None):
        """
        Returns logits (and mixed y during training if mixup is active).
        """
        # ── embed ─────────────────────────────────────────────────────────────
        parts = [self.num_embed(x_num)] # [B, Cn, D]
        if self.cat_embed is not None and x_cat is not None:
            parts.append(self.cat_embed(x_cat)) # [B, Cc, D]
        x = torch.cat(parts, dim=1) # [B, C, D]

        # ── Feat-Mix augmentation ─────────────────────────────────────────────
        mixed_y = y
        if self.mixup == "feat_mix" and y is not None:
            x, mixed_y, _ = self._feat_mix(x, y)

        # ── transformer blocks ────────────────────────────────────────────────
        for i, block in enumerate(self.blocks):
            # Hidden-Mix applied after 1st block
            if self.mixup == "hidden_mix" and i == 1 and y is not None:
                x, mixed_y, _ = self._hidden_mix(x, y)
            x = block(x, self.spa_mask)

        # ── attentive decoder ─────────────────────────────────────────────────
        x = self.ln_out(x) # [B, C, D]
        w = torch.softmax(self.attn_w(x), dim=1) # [B, C, 1]
        out = (w * x).sum(dim=1) # [B, D]
        logits = self.head(out) # [B, out_channels]

        if y is not None and self.training and self.mixup is not None and isinstance(mixed_y, tuple):
            return logits, mixed_y
        return logits, None


# ─────────────────────────────────────────────────────────────────────────────
# 3. MIXUP LOSS (BCE-based, single logit)
# ─────────────────────────────────────────────────────────────────────────────

def mixup_bce(logits: Tensor, mixed_y, weights: Optional[Tensor] = None) -> Tensor:
    """
    Binary cross-entropy mixup loss.
    mixed_y is either:
      • a plain float tensor → standard weighted BCE
      • a tuple (y_a, y_b, lam) → interpolated BCE
    """
    if isinstance(mixed_y, tuple):
        y_a, y_b, lam = mixed_y
        loss_a = F.binary_cross_entropy_with_logits(logits, y_a.float(), weight=weights)
        loss_b = F.binary_cross_entropy_with_logits(logits, y_b.float(), weight=weights)
        return lam * loss_a + (1 - lam) * loss_b
    return F.binary_cross_entropy_with_logits(logits, mixed_y.float(), weight=weights)

In [ ]:

def mixup_bce(logits: Tensor, mixed_y, weights: Optional[Tensor] = None) -> Tensor:
    """
    Binary cross-entropy mixup loss.
    mixed_y is either:
      • a plain float tensor → standard weighted BCE
      • a tuple (y_a, y_b, lam) → interpolated BCE
    """
    if isinstance(mixed_y, tuple):
        y_a, y_b, lam = mixed_y
        loss_a = F.binary_cross_entropy_with_logits(logits, y_a.float(), weight=weights)
        loss_b = F.binary_cross_entropy_with_logits(logits, y_b.float(), weight=weights)
        return lam * loss_a + (1 - lam) * loss_b
    return F.binary_cross_entropy_with_logits(logits, mixed_y.float(), weight=weights)

## 5. Training Constants

In [ ]:
BATCH_SIZE = 1024 # 8 192 — same as reference
N_EPOCHS = 100
PATIENCE = 10
POS_WEIGHT = 3.5 # class weight for positive class
# PARQUET_PATH = "processed_data_densest.parquet.gzip" # real file or synthetic
RESULTS_JSON = str(RESULTS_DIR / "excelformer_lc_trials.json")
GLOBAL_MODEL_PATH = str(MODELS_DIR / "excelformer_lc_best.pth")
import json
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
trial_results: List[dict] = []
global_best_auc = 0.73

## 6. Optuna Optimization

In [ ]:
def optim(trial: optuna.Trial) -> float:
    global global_best_auc

    # ── search space ──────────────────────────────────────────────────────────
    d_token = trial.suggest_categorical("d_token", [64, 128])
    n_blocks = trial.suggest_int( "n_blocks", 3, 5)
    attention_heads = trial.suggest_categorical("attn_heads", [4, 8])
    diam_dropout = trial.suggest_float( "diam_drop", 0.0, 0.2)
    aium_dropout = trial.suggest_float( "aium_drop", 0.0, 0.2)
    residual_dropout = trial.suggest_float( "res_drop", 0.0, 0.1)
    mixup_type = trial.suggest_categorical("mixup", ["feat_mix", "hidden_mix", "none"])
    lr = trial.suggest_float( "lr", 1e-4, 5e-3, log=True)

    if mixup_type == "none":
        mixup_type = None

    # ── print trial header (mirrors reference style) ──────────────────────────
    print("\n" + "=" * 70)
    print(f" Starting Trial {trial.number}")
    print(" Hyperparameters:")
    print(f" d_token = {d_token}")
    print(f" n_blocks = {n_blocks}")
    print(f" attention_heads = {attention_heads}")
    print(f" diam_dropout = {diam_dropout:.3f}")
    print(f" aium_dropout = {aium_dropout:.3f}")
    print(f" residual_dropout = {residual_dropout:.3f}")
    print(f" mixup = {mixup_type}")
    print(f" lr = {lr:.2e}")
    print("=" * 70 + "\n")

    # ── model ─────────────────────────────────────────────────────────────────
    model = ExcelFormer(
        num_numerical = d_numerical,
        cat_cardinalities= cat_cardinalities,
        embed_dim = d_token,
        num_layers = n_blocks,
        num_heads = attention_heads,
        out_channels = 1, # single logit → BCE
        diam_dropout = diam_dropout,
        aium_dropout = aium_dropout,
        residual_dropout = residual_dropout,
        mixup = mixup_type,
        beta = 0.5,
    ).to(device)

    n_params = sum(p.numel() for p in model.parameters() if p.requires_grad)
    print(f"Parameters: {n_params:,}\n")

    # ── optimizer + OneCycleLR (per-batch step, mirrors reference) ────────────
    epoch_size = math.ceil(n_train / BATCH_SIZE)
    optimizer = torch.optim.AdamW(model.parameters(), lr=lr, weight_decay=1e-5)
    scheduler = torch.optim.lr_scheduler.OneCycleLR(
        optimizer,
        max_lr = lr * 5, # 5× base, same ratio as reference
        steps_per_epoch = epoch_size,
        epochs = N_EPOCHS,
        pct_start = 0.3,
        anneal_strategy = "cos",
    )

    loss_fn = F.binary_cross_entropy_with_logits

    # ── apply_model closure (mirrors reference exactly) ───────────────────────
    def apply_model(batch: Dict[str, Tensor]) -> Tensor:
        logits, _ = model(batch["X_cont"], batch.get("x_cat"))
        return logits.squeeze(-1) # [B] — single logit per sample

    # ── evaluate closure (mirrors reference exactly) ──────────────────────────
    @torch.no_grad()
    def evaluate(part: str) -> float:
        model.eval()
        y_pred = (
            torch.cat(
                [apply_model(batch)
                 for batch in delu.iter_batches(data[part], BATCH_SIZE)]
            )
            .cpu()
            .numpy()
        )
        y_true = data[part]["y"].cpu().numpy()
        y_pred = scipy.special.expit(y_pred) # sigmoid → probability
        return roc_auc_score(y_true, y_pred)

    # ── pre-training baseline ─────────────────────────────────────────────────
    print(f"Test score before training: AUC Val = {evaluate('val'):.4f}")
    print(f"Device: {device.type.upper()}")
    print("-" * 88 + "\n")

    # ── delu early stopping + timer ───────────────────────────────────────────
    timer = delu.tools.Timer()
    early_stopping = delu.tools.EarlyStopping(PATIENCE, mode="max")
    best = {"val": -math.inf, "test": -math.inf, "epoch": -1}

    timer.run()

    # ── training loop ─────────────────────────────────────────────────────────
    for epoch in range(N_EPOCHS):

        for batch in tqdm(
            delu.iter_batches(data["train"], BATCH_SIZE, shuffle=True),
            desc=f"Epoch {epoch}",
            total=epoch_size,
        ):
            model.train()
            optimizer.zero_grad()

            y_batch = batch["y"].float()

            # Class weights — mirrors reference's weighted BCE
            weights = torch.where(
                y_batch == 1,
                torch.tensor(POS_WEIGHT, device=y_batch.device),
                torch.tensor(1.0, device=y_batch.device),
            )

            raw_logits = apply_model(batch)

            # Mixup augmentation (if active the model returns mixed_y)
            logits, mixed_y = model(batch["X_cont"], batch.get("x_cat"), y_batch)
            logits = logits.squeeze(-1)

            if mixed_y is not None:
                loss = mixup_bce(logits, mixed_y, weights=None) # weights inside mixup
            else:
                loss = loss_fn(logits, y_batch, weight=weights)

            loss.backward()
            nn.utils.clip_grad_norm_(model.parameters(), 1.0)
            optimizer.step()
            scheduler.step() # ← per-batch, same as reference

        # ── epoch-end metrics ─────────────────────────────────────────────────
        val_auc = evaluate("val")
        test_auc = evaluate("test")
        print(f" AUC Val {val_auc:.4f} Test {test_auc:.4f} [time] {timer}")

        trial.report(val_auc, step=epoch)
        if trial.should_prune():
            print(f" ️ Trial {trial.number} pruned at epoch {epoch}")
            trial_results.append({
                "trial_number": trial.number,
                "trial_type": "ExcelFormer SPA + AIUM",
                "auc_val": val_auc,
                "auc_test": test_auc,
                "d_token": d_token,
                "n_blocks": n_blocks,
                "attn_heads": attention_heads,
                "diam_dropout": diam_dropout,
                "aium_dropout": aium_dropout,
                "residual_dropout":residual_dropout,
                "mixup": str(mixup_type),
                "lr": lr,
                "best": best,
                "time": str(timer),
            })
            with open(RESULTS_JSON, "w") as f:
                json.dump(trial_results, f, indent=4)
            raise optuna.exceptions.TrialPruned()

        # ── early stopping ────────────────────────────────────────────────────
        early_stopping.update(val_auc)
        if early_stopping.should_stop():
            print("⏹ Early stopping triggered.")
            break

        # ── checkpoint (global guard, mirrors reference) ──────────────────────
        if val_auc > best["val"]:
            print(" New best epoch! ")
            best = {"val": val_auc, "test": test_auc, "epoch": epoch}

            if val_auc > global_best_auc:
                torch.save(model.state_dict(), GLOBAL_MODEL_PATH)
                global_best_auc = val_auc
                print(f" Saved to {GLOBAL_MODEL_PATH} (global best {global_best_auc:.4f})")

    # ── persist trial result ──────────────────────────────────────────────────
    trial_results.append({
        "trial_number": trial.number,
        "trial_type": "ExcelFormer SPA + AIUM",
        "auc_val": val_auc,
        "auc_test": test_auc,
        "d_token": d_token,
        "n_blocks": n_blocks,
        "attn_heads": attention_heads,
        "diam_dropout": diam_dropout,
        "aium_dropout": aium_dropout,
        "residual_dropout":residual_dropout,
        "mixup": str(mixup_type),
        "lr": lr,
        "best": best,
        "time": str(timer),
    })
    with open(RESULTS_JSON, "w") as f:
        json.dump(trial_results, f, indent=4)

    print("\n\nResult:")
    print(best)
    return best["val"]
pruner = optuna.pruners.MedianPruner(
    n_startup_trials=3,
    n_warmup_steps=int(N_EPOCHS * 0.3) + 5,
    interval_steps=3,
)
study = optuna.create_study(direction="maximize",
                            study_name="excelformer_lendingclub",
                            pruner=pruner)
study.optimize(optim, n_trials=(1 if DEV_MODE else 30))